# Twitter Sentiment Analysis
**Input:** `data/processed/twitter_processed/twitter_clean.csv`  
**Output:** `data/processed/twitter_processed/twitter_with_sentiment.csv` + figures

**Models used:**
1. **VADER** — rule-based, fast, good baseline
2. **RoBERTa** (`cardiffnlp/twitter-roberta-base-sentiment-latest`) — transformer fine-tuned *on Twitter data*, best accuracy

Coverage:
- Sentiment distributions (VADER vs RoBERTa)
- Monthly sentiment trend
- Pre/post ChatGPT sentiment shift
- Sentiment vs engagement
- Model agreement analysis

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from src.twitter_data_prep import aggregate_by_month
from src.sentiment import vader_sentiment, roberta_sentiment, compute_agreement, flag_sentiment_quality

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

OUT_FIG  = os.path.abspath('../../reports/twitter_reports')
OUT_DATA = os.path.abspath('../../data/processed/twitter_processed')
os.makedirs(OUT_FIG,  exist_ok=True)
os.makedirs(OUT_DATA, exist_ok=True)
print('Setup OK')

## 1. Load Clean Data

In [ ]:
df = pd.read_csv('../../data/processed/twitter_processed/twitter_clean.csv', parse_dates=['date'])
print(f'Loaded: {df.shape}')

# Use cleaned_text for ML — same as combined_text in reddit pipeline
df = df.rename(columns={'cleaned_text': 'combined_text'})
df.head(3)

## 2. VADER Sentiment
Rule-based, instantaneous, no GPU needed.

In [ ]:
print('Running VADER...')
df = vader_sentiment(df, text_col='combined_text')
print('VADER done.')
print(df['vader_label'].value_counts())

## 3. RoBERTa Sentiment
**cardiffnlp/twitter-roberta-base-sentiment-latest** — specifically trained on tweet data.

> ⚠️ This will download ~500MB on first run. Use GPU if available. CPU will be slower but works.

In [ ]:
print(f'Running RoBERTa on {len(df)} tweets...')
df = roberta_sentiment(df, text_col='combined_text', batch_size=32, max_chars=280)
print('RoBERTa done.')
print(df['roberta_label'].value_counts())

## 4. Model Agreement

In [ ]:
df = flag_sentiment_quality(df)
df = compute_agreement(df)

agree_rate = df['vader_roberta_agree'].mean() * 100
print(f'VADER–RoBERTa agreement: {agree_rate:.1f}%')
print()
print('Disagreement breakdown:')
print(df[~df['vader_roberta_agree']]['disagreement_type'].value_counts())

## 5. Sentiment Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
order   = ['positive', 'neutral', 'negative']
palette = {'positive': '#4CAF50', 'neutral': '#FFC107', 'negative': '#F44336'}

for ax, (col, title) in zip(axes, [('vader_label','VADER'), ('roberta_label','RoBERTa')]):
    counts = df[col].value_counts().reindex(order)
    bars = ax.bar(order, counts, color=[palette[o] for o in order], edgecolor='none', width=0.5)
    for bar, v in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + counts.max()*0.01,
                f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=9)
    ax.set_title(f'{title} Sentiment Distribution', fontsize=12, weight='bold')
    ax.set_xlabel('Sentiment')
    ax.set_ylabel('Tweet Count')

plt.suptitle('Sentiment Distributions — VADER vs RoBERTa', fontsize=13, weight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT_FIG}/tw_sentiment_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Monthly Sentiment Trend

In [ ]:
from src.sentiment import aggregate_sentiment_by_month

monthly_sent = aggregate_sentiment_by_month(df)
reliable = monthly_sent[monthly_sent['reliable']]

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Top: stacked area chart of sentiment %
ax = axes[0]
ax.stackplot(
    reliable['month'],
    reliable['positive_pct'], reliable['neutral_pct'], reliable['negative_pct'],
    labels=['Positive', 'Neutral', 'Negative'],
    colors=['#4CAF50', '#FFC107', '#F44336'],
    alpha=0.85
)
ax.axvline(pd.Timestamp('2022-11-30'), color='black', ls='--', lw=1.5, label='ChatGPT launch')
ax.set_title('Monthly Sentiment Composition (%)', fontsize=12, weight='bold')
ax.set_ylabel('% of Tweets')
ax.legend(loc='upper left', fontsize=9)
ax.set_ylim(0, 100)

# Bottom: RoBERTa continuous score
ax2 = axes[1]
ax2.plot(reliable['month'], reliable['roberta_continuous_mean'],
         color='#1565C0', lw=2, label='RoBERTa score')
ax2.axhline(0, color='grey', ls=':', lw=1)
ax2.axvline(pd.Timestamp('2022-11-30'), color='black', ls='--', lw=1.5)
ax2.fill_between(reliable['month'], reliable['roberta_continuous_mean'], 0,
                  where=(reliable['roberta_continuous_mean'] >= 0),
                  color='#4CAF50', alpha=0.3, label='Net positive')
ax2.fill_between(reliable['month'], reliable['roberta_continuous_mean'], 0,
                  where=(reliable['roberta_continuous_mean'] < 0),
                  color='#F44336', alpha=0.3, label='Net negative')
ax2.set_title('Monthly RoBERTa Sentiment Score', fontsize=12, weight='bold')
ax2.set_ylabel('Score (pos=1, neg=-1)')
ax2.set_xlabel('Month')
ax2.legend(fontsize=9)
ax2.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,7]))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)

plt.tight_layout()
plt.savefig(f'{OUT_FIG}/tw_monthly_sentiment.png', dpi=150)
plt.show()

## 7. Pre/Post ChatGPT Sentiment Shift

In [ ]:
period_sent = df.groupby(['period','roberta_label']).size().unstack(fill_value=0)
period_pct  = period_sent.div(period_sent.sum(axis=1), axis=0) * 100

print('Sentiment % by period:')
print(period_pct.round(1))

fig, ax = plt.subplots(figsize=(8, 5))
period_pct.reindex(['pre-ChatGPT','post-ChatGPT'])[['positive','neutral','negative']].plot(
    kind='bar', ax=ax,
    color=['#4CAF50', '#FFC107', '#F44336'],
    edgecolor='none', width=0.5
)
ax.set_title('Sentiment Distribution: Pre vs Post ChatGPT Launch', fontsize=13, weight='bold')
ax.set_ylabel('% of Tweets')
ax.set_xlabel('')
ax.set_xticklabels(['pre-ChatGPT', 'post-ChatGPT'], rotation=0)
ax.legend(title='Sentiment')
plt.tight_layout()
plt.savefig(f'{OUT_FIG}/tw_pre_post_sentiment.png', dpi=150)
plt.show()

## 8. Sentiment vs Engagement

In [ ]:
eng_by_sent = df.groupby('roberta_label').agg(
    avg_likes      = ('likes',           'mean'),
    avg_retweets   = ('retweets',        'mean'),
    avg_engagement = ('total_engagement','mean'),
    tweet_count    = ('id',              'count'),
).round(2).reindex(['positive','neutral','negative'])

print(eng_by_sent)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
metrics = ['avg_likes', 'avg_retweets', 'avg_engagement']
titles  = ['Avg Likes', 'Avg Retweets', 'Avg Total Engagement']
colors  = ['#4CAF50', '#FFC107', '#F44336']

for ax, metric, title in zip(axes, metrics, titles):
    vals = eng_by_sent[metric]
    bars = ax.bar(vals.index, vals.values, color=colors, edgecolor='none', width=0.5)
    for bar, v in zip(bars, vals.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+vals.max()*0.01,
                f'{v:.1f}', ha='center', fontsize=10)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Average')

plt.suptitle('Engagement by Sentiment Label (RoBERTa)', fontsize=13, weight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT_FIG}/tw_sentiment_vs_engagement.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Save Data with Sentiment

In [ ]:
out = os.path.join(OUT_DATA, 'twitter_with_sentiment.csv')
df.to_csv(out, index=False)
print(f'Saved → {out}')
print(f'Columns: {df.columns.tolist()}')